In [1]:
import json
from transformers import AutoProcessor
import sys 
import os 
# current_file_path = os.path.dirname(os.path.abspath(__file__))
# module_path = os.path.join(current_file_path, "../")
# sys.path.append(module_path)
# from models.qwen2_5_vl import Qwen2VLRetForConditionalGeneration
import torch 
import argparse
from dataset.datasets_mbeir import QueryDataset, CandidateDataset
from collators.mbeir_eval import MbeirQueryDataCollator, MbeirCandidateDataCollator
from torch.utils.data import DataLoader 
import torch.nn.functional as F 
import numpy as np
DATASET_QUERY_NUM_UPPER_BOUND = 500000
DATASET_CAN_NUM_UPPER_BOUND = 10000000

NUM_QUERIES = 30
NUM_CANDIDATES_FOR_POOL = 300 # Number of candidates to use in the evaluation pool for simulated retrieval
MAX_RETRIEVED_PER_QUERY = 50 # For simulated retrieval and recall calculation up to K=50


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:105: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [5]:

def unhash_qid(hashed_qid):
    dataset_id = hashed_qid // DATASET_QUERY_NUM_UPPER_BOUND
    data_within_id = hashed_qid % DATASET_QUERY_NUM_UPPER_BOUND
    return f"{dataset_id}:{data_within_id}"

def unhash_did(hashed_did):
    dataset_id = hashed_did // DATASET_CAN_NUM_UPPER_BOUND
    data_within_id = hashed_did % DATASET_CAN_NUM_UPPER_BOUND
    return f"{dataset_id}:{data_within_id}"

def load_qrel(filename):
    qrel = {}
    qid_to_taskid = {}
    with open(filename, "r") as f:
        for line in f:
            query_id, _, doc_id, relevance_score, task_id = line.strip().split()
            if int(relevance_score) > 0:  # Assuming only positive relevance scores indicate relevant documents
                if query_id not in qrel:
                    qrel[query_id] = []
                qrel[query_id].append(doc_id)
                if query_id not in qid_to_taskid:
                    qid_to_taskid[query_id] = task_id
    print(f"Retriever: Loaded {len(qrel)} queries from {filename}")
    print(
        f"Retriever: Average number of relevant documents per query: {sum(len(v) for v in qrel.values()) / len(qrel):.2f}"
    )
    return qrel, qid_to_taskid

def compute_recall_at_k(relevant_docs, retrieved_indices, k):
    if not relevant_docs:
        return 0.0 # Return 0 if there are no relevant documents

    # Get the set of indices for the top k retrieved documents
    top_k_retrieved_indices_set = set(retrieved_indices[:k])

    # Convert the relevant documents to a set
    relevant_docs_set = set(relevant_docs)

    # Check if there is an intersection between relevant docs and top k retrieved docs
    # If there is, we return 1, indicating successful retrieval; otherwise, we return 0
    if relevant_docs_set.intersection(top_k_retrieved_indices_set):
        return 1.0
    else:
        return 0.0



In [3]:

class Args:
    def __init__(self):
        # Define the environment variables from the command
        _MODEL_ID = "./checkpoints/LamRA-Ret"
        _ORIGINAL_MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"
        _IMAGE_PATH_PREFIX = "/mnt/tidalfs-hssh01/dataset/mmeb/M-BEIR"

        # Arguments passed in the command line
        self.query_data_path: str = f"{_IMAGE_PATH_PREFIX}/query/test/mbeir_xhs_task7_test.jsonl"
        self.query_cand_pool_path: str = f"{_IMAGE_PATH_PREFIX}/cand_pool/local/mbeir_xhs_task7_cand_pool.jsonl"
        self.cand_pool_path: str = f"{_IMAGE_PATH_PREFIX}/cand_pool/local/mbeir_xhs_task7_cand_pool.jsonl"
        self.instructions_path: str = f"{_IMAGE_PATH_PREFIX}/instructions/query_instructions.tsv"
        self.qrels_path: str = f"{_IMAGE_PATH_PREFIX}/qrels/test/mbeir_xhs_task7_test_qrels.txt"
        self.original_model_id: str = _ORIGINAL_MODEL_ID
        self.image_path_prefix: str = _IMAGE_PATH_PREFIX
        self.model_id: str = _MODEL_ID

        # Argument with a default value from the argparse definition (not overridden in the command)
        self.model_max_length: int = 1024

# xhs 评估指标验证

In [6]:
args = Args()
from dataset.datasets_mbeir import QueryDataset, CandidateDataset
from PIL import Image
cand_dataset = CandidateDataset(
    query_data_path=args.query_data_path, 
    cand_pool_path=args.cand_pool_path,
    instructions_path=args.instructions_path,
    image_path_prefix=args.image_path_prefix
)
query_dataset = QueryDataset(
    query_data_path=args.query_data_path, 
    cand_pool_path=args.query_cand_pool_path,
    instructions_path=args.instructions_path,
    image_path_prefix=args.image_path_prefix
)


In [7]:
qrel, _ = load_qrel(args.qrels_path)
cand_pool = {d['did']:d for d in cand_dataset.cand_pool}

Retriever: Loaded 999 queries from /mnt/tidalfs-hssh01/dataset/mmeb/M-BEIR/qrels/test/mbeir_xhs_task7_test_qrels.txt
Retriever: Average number of relevant documents per query: 4.24


In [55]:

def show_some_query_info(query_idx):
    query, _ = query_dataset[query_idx]
    qimg = query[0]['content'][0]['image']
    qbox = query[0]['content'][0]['box']
    # cands = cand_names_json[query_idx] # len 50
    poscand_did_list = qrel[f'10:{query_idx+1}']
    # print(poscand_did)
    # for i in (1,5,10,50):
    #     for poscand_did in poscand_did_list:
    #         if poscand_did in cands[:i]:
    #             print(f"right in recall@{i}")
    #             break
    imglist = [qimg]
    box_list = [qbox]
    print(qbox)
    imglist.extend([ cand_pool[did]['img_path'] for did in poscand_did_list])
    box_list.extend([ cand_pool[did]['box'] for did in poscand_did_list])
    
    show_group_imgs(imglist,box_list)
    return query, poscand_did_list


def show_some_cand(did):
    some_cand = cand_pool[did]
    return Image.open(some_cand['img_path'])

def show_group_imgs(image_paths, box_list=None, output_path=None):
    def draw(image, box):
        from PIL import Image, ImageDraw
        if box is None: return image
        width, height = image.size
        x0 = width * box[0]
        y0 = height * box[1]
        x1 = width * box[2]
        y1 = height * box[3]

        image = image.crop((x0, y0, x1, y1))
        return image

        
        draw = ImageDraw.Draw(image)
        # (x0, y0) is top-left, (x1, y1) is bottom-right
        line_thickness = max(2, int(min(width, height) * 0.01))
        draw.rectangle(
            [(x0, y0), (x1, y1)], 
            outline="red", 
            width=line_thickness
        )
        return image
    images = []
    if box_list is None: box_list = [None] * len(image_paths)
    for path, box in zip(image_paths, box_list):
        images.append(draw(Image.open(path), box).resize((224,224)))
    
    # 获取第一张图片的模式和大小
    mode = images[0].mode
    width, height = images[0].size
    
    # 检查所有图片是否模式一致
    for img in images:
        if img.mode != mode:
            raise ValueError("所有图片必须具有相同的模式")
    
    # 计算拼接后总宽度和高度
    total_width = sum(img.width for img in images)
    max_height = max(img.height for img in images)
    
    # 创建空白画布
    result = Image.new(mode, (total_width, max_height), (0, 0, 0, 0))
    
    # 拼接图片
    x_offset = 0
    for img in images:
        result.paste(img, (x_offset, 0))
        x_offset += img.width
    
    # 显示结果
    result.show()

# def diff_topk_cands(ours, baseline, topk=5):
#     strong, weak = [] , []
#     for query_idx in range(len(query_dataset)):
#         # for t in qrel[f'10:{query_idx+1}']:
#         t = np.random.choice(qrel[f'10:{query_idx+1}'])
#         if t in ours[query_idx][:topk] and (t not in baseline[query_idx][:topk]):
#             strong.append(query_idx)
#         elif t not in ours[query_idx][:topk] and (t in baseline[query_idx][:topk]):
#             weak.append(query_idx)
#     print(f"strong:{len(strong)}, weak:{len(weak)}")
#     return strong, weak

def diff_topk_cands(ours, baseline, topk=5):
    strong, weak, mid = [] , [], []
    totalq_len = 0
    ourscnt, baselinecnt = 0, 0
    for query_idx in range(len(query_dataset)):
        cset = set(qrel[f'10:{query_idx+1}'])
        totalq_len += len(cset)
        ourset = set(ours[query_idx])
        baseset = set(baseline[query_idx])
        for t in cset:
            extras = cset - {t}
            l1 = len(ourset.intersection(extras))
            l2 = len(baseset.intersection(extras))
            t_in_ours = t in set(ours[query_idx][:topk+l1])
            t_in_baseline = t in set(baseline[query_idx][:topk+l2])
            if t_in_ours: ourscnt += 1
            if t_in_baseline: baselinecnt += 1
            if t_in_ours and not t_in_baseline:
                strong.append(query_idx)
            elif not t_in_ours and t_in_baseline:
                weak.append(query_idx)
            elif t_in_ours and t_in_baseline:
                mid.append(query_idx)
    print(f"strong:{len(strong)}, weak:{len(weak)}, both:{len(mid)}")
    print(f"ours acc:{ourscnt/totalq_len}, baseline acc:{baselinecnt/totalq_len}")
    return strong, weak

def compute_recall_at_k(relevant_docs, retrieved_indices, k):
    if not relevant_docs:
        return 0.0 # Return 0 if there are no relevant documents

    # Get the set of indices for the top k retrieved documents
    top_k_retrieved_indices_set = set(retrieved_indices[:k])

    # Convert the relevant documents to a set
    relevant_docs_set = set(relevant_docs)

    # Check if there is an intersection between relevant docs and top k retrieved docs
    # If there is, we return 1, indicating successful retrieval; otherwise, we return 0
    result = []
    for x in relevant_docs_set:
        filtered_top_k = top_k_retrieved_indices_set - (relevant_docs_set - {x})
        if x in filtered_top_k:
            result.append(1.0)
        else:
            result.append(0.0)
    return result


In [30]:
# 展示poscand
# print(poscand_did_list)
# show_some_cand(poscand_did_list[0])

# 第一组实验，发现效果变差了  note-1k-1top-1pos-filter

In [24]:
!ls LamRA_Ret_eval_results | grep qwen2_5-vl-7b_cvp+xhs_ablation_lemuir_results

qwen2_5-vl-7b_cvp+xhs_ablation_lemuir_results.txt


In [25]:
!cat LamRA_Ret_eval_results/qwen2_5-vl-7b_cvp+xhs_ablation_lemuir_results.txt

/mnt/tidal-alsh01/dataset/mmeb/M-BEIR/qrels/test/mbeir_xhs_task7_test_qrels.txt
recall_at_1 = 0.1854648419065597
recall_at_5 = 0.6753185464841907
recall_at_10 = 0.9023124115148655
recall_at_50 = 0.9827748938178386


In [58]:
rootdir = "LamRA_Ret_eval_results"

with open("LamRA_Ret_eval_results/mbeir_xhs_task7_test_qwen2_5-vl-7b_candelete_cand_names.json", "r") as f:
    ourscand_names_json = json.load(f)

with open("LamRA_Ret_eval_results/mbeir_xhs_task7_test_qwen2_5-vl-7b_cvp+xhs_ablation_lemuir_cand_names.json", "r") as f:
    baselinecand_names_json = json.load(f)    

In [17]:
strong, weak = diff_topk_cands(ourscand_names_json, baselinecand_names_json, topk=1)

strong:526, weak:537, both:2436
ours acc:0.6989145823501651, baseline acc:0.7015101462954224


In [ ]:
# 展示top 5 的cands
# show_some_cand(cands[4])
# 如果是ours
query_idx = weak[8]
show_some_query_info(query_idx)
print("our results")

ourcands = ourscand_names_json[query_idx] # len 50
show_group_imgs([cand_pool[ourcands[i]]['img_path'] for i in range(10)],box_list=[cand_pool[ourcands[i]]['box'] for i in range(10)])
# print("baseline results")
# baselinecands = baselinecand_names_json[query_idx]
# show_group_imgs([cand_pool[baselinecands[i]]['img_path'] for i in range(10)])

In [70]:
# cand_pool[ourcands[1]]['box']